In [0]:
# Databricks notebook source
# ============================================
# OPENBREWERYDB - LANDING LAYER INGESTION
# ============================================

# COMMAND ----------

# ============================================
# WIDGETS
# ============================================

dbutils.widgets.text("catalog", "brewing")
dbutils.widgets.text("schema", "landing")
dbutils.widgets.text("volume_path", "/Volumes/brewing/landing/openbrewerydb/raw")
dbutils.widgets.text("base_url", "https://api.openbrewerydb.org/v1/breweries")
dbutils.widgets.text("per_page", "200")
dbutils.widgets.text("max_pages", "100")



In [0]:
# COMMAND ----------

# ============================================
# IMPORTS
# ============================================

import requests
import json
import time
from datetime import datetime
from pyspark.sql import SparkSession

# COMMAND ----------

# ============================================
# PARAMETERS
# ============================================

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

VOLUME_PATH = dbutils.widgets.get("volume_path")
BASE_URL = dbutils.widgets.get("base_url")

PER_PAGE = int(dbutils.widgets.get("per_page"))
MAX_PAGES = int(dbutils.widgets.get("max_pages"))

REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
RETRY_DELAY = 5

# COMMAND ----------

# ============================================
# EXECUTION METADATA
# ============================================

execution_timestamp = datetime.utcnow()
execution_date = execution_timestamp.strftime("%Y-%m-%d")
execution_datetime = execution_timestamp.strftime("%Y%m%d_%H%M%S")

landing_path = f"{VOLUME_PATH}/ingestion_date={execution_date}"

print("=" * 60)
print("STARTING LANDING INGESTION")
print(f"Execution Timestamp: {execution_timestamp}")
print(f"Landing Path: {landing_path}")
print("=" * 60)

# COMMAND ----------

# ============================================
# CREATE DIRECTORY IF NOT EXISTS
# ============================================

dbutils.fs.mkdirs(landing_path)

# COMMAND ----------



In [0]:
# ============================================
# HTTP REQUEST FUNCTION
# ============================================

def make_request(url, params=None):

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = requests.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT
            )

            if response.status_code == 200:
                return response.json()

            elif response.status_code == 429:
                print(f"Rate limit exceeded. Retry {attempt}/{MAX_RETRIES}")
                time.sleep(RETRY_DELAY)

            elif response.status_code >= 500:
                print(f"Server error {response.status_code}. Retry {attempt}/{MAX_RETRIES}")
                time.sleep(RETRY_DELAY)

            else:
                raise Exception(
                    f"Request failed with status code {response.status_code}"
                )

        except requests.exceptions.Timeout:
            print(f"Timeout occurred. Retry {attempt}/{MAX_RETRIES}")
            time.sleep(RETRY_DELAY)

        except requests.exceptions.ConnectionError:
            print(f"Connection error. Retry {attempt}/{MAX_RETRIES}")
            time.sleep(RETRY_DELAY)

        except Exception as e:
            print(f"Unexpected error: {str(e)}")
            time.sleep(RETRY_DELAY)

    raise Exception("Max retries exceeded")



In [0]:
# COMMAND ----------

# ============================================
# PAGINATED INGESTION
# ============================================

total_records = 0
total_pages = 0

for page in range(1, MAX_PAGES + 1):

    print(f"Processing page {page}")

    params = {
        "page": page,
        "per_page": PER_PAGE
    }

    data = make_request(BASE_URL, params=params)

    # Stop when API returns empty page
    if not data:
        print(f"No more records found at page {page}")
        break

    # Add ingestion metadata
    enriched_data = {
        "metadata": {
            "source": "openbrewerydb",
            "ingestion_timestamp_utc": execution_timestamp.isoformat(),
            "page": page,
            "records": len(data)
        },
        "data": data
    }

    file_name = f"page_{str(page).zfill(4)}_{execution_datetime}.json"

    output_path = (
    f"{landing_path}/{file_name}"
)

    json_content = json.dumps(
        enriched_data,
        ensure_ascii=False,
        indent=2
    )

    dbutils.fs.put(
        output_path,
        json_content,
        overwrite=True
    )

    print(f"Saved file: {output_path}")

    total_records += len(data)
    total_pages += 1



In [0]:
# COMMAND ----------

# ============================================
# EXECUTION SUMMARY
# ============================================

print("=" * 60)
print("INGESTION FINISHED")
print(f"Total Pages Processed: {total_pages}")
print(f"Total Records Ingested: {total_records}")
print(f"Landing Path: {landing_path}")
print("=" * 60)